# 1. Import Required Libraries
Import necessary libraries for data manipulation and spatial operations.

In [ ]:
import pandas as pd
import geopandas as gpd

# 2. Load Raw Address Data
Load the raw Milwaukee address points or street centerlines dataset.

In [ ]:
# load ArcGIS address points data from zipped shapefile
arcgis = gpd.read_file("Address_Points.zip")
arcgis.head()

# 3. Standardize Address Formats
Clean and filter the dataset to get a standardized 'FullAddr' column that matches the format expected by the ticket dataset. We only want addresses explicitly assigned to the City of Milwaukee to limit False matches out in the suburbs.

In [ ]:
# group by municipal and only take 'Milwaukee' addresses
munis = arcgis.groupby("Muni")
milwaukee = munis.get_group("Milwaukee")

# the FullAddr column contains what we need for tickets, we extract it alongside the geometry
milwaukee = milwaukee[['FullAddr', 'geometry']]
milwaukee.reset_index(drop=True, inplace=True)
milwaukee.head()

# 4. Process Spatial Geometry
Ensure the geometry is correctly formatted as points (converting geometric types if necessary), handle any missing geometries, and prepare the dataset for standard coordinate extraction.

In [ ]:
# filter out any points lacking physical geometry or Address names
milwaukee = milwaukee.dropna(subset=['FullAddr', 'geometry'])

# the geometry holds the shapely POINT mapping in WKT natively as strings when dumped to CSV. 
print(f"Total Clean Addresses for Milwaukee Geocoding: {len(milwaukee)}")

# 5. Export Cleaned Address Points
Save the processed DataFrame to 'data/Address_Points_Clean.csv' for use in the main ticket geocoding pipeline.

In [ ]:
# export to csv in the core data folder without indexing
milwaukee.to_csv("data/Address_Points_Clean.csv", index=False)
print("Saved geometry data to data/Address_Points_Clean.csv")